# Guerilla Search — 上下文感知网站分析

## 练习目标（理念）

**Guerilla Search** 是一款由 LLM 支持的网站分析工具，目标不止于传统摘要。

系统不会简单浓缩网页内容，而是探索网站呈现内容背后更广泛的背景：给定 URL，抓取网站内容，并生成能区分「显式表述」与「推断上下文」的结构化分析。

## 项目亮点

* **官方故事（Official Story）** — 网站明确传达的信息
* **隐藏背景（Hidden Context）** — 可能的假设、激励与框架
* **缺失的声音（Missing Voices）** — 可能未被代表的观点
* **公众反应（Public Reaction）** — 来自不同受众的假设性观点
* **游击裁决（Guerrilla Verdict）** — 简洁、诙谐的最终观察

系统鼓励批判性阅读：把事实与推论分开，并避免缺乏依据的主张。本项目演示 LLM 如何从「概括」走向「上下文解释与视角感知分析」。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `fetch_website_contents(url)` |
| Chat Completions | `client.chat.completions.create(...)` |
| system / user messages | `messages_for(context)` |
| 非 OpenAI 后端 | Gemini 的 OpenAI 兼容端点 |

## 怎么跑

1. 准备 `.env`：配置 `GOOGLE_API_KEY`
2. 从上到下依次运行单元格（Shift+Enter）
3. 最后一个单元格改 URL，即可分析别的网站


In [8]:
# ========== 导入：后面要用的工具箱 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 GOOGLE_API_KEY
import os
# 导入标准库 sys：这里用来改 Python 模块搜索路径 sys.path
import sys
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境，避免把密钥写进代码
from dotenv import load_dotenv
# 把仓库更上层目录加入 path，以便 import week1.scraper（相对本笔记本的路径）
sys.path.append("../..")
# 从课程 week1 的 scraper 导入抓取函数：取网页正文文本
from week1.scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里用 Markdown 漂亮显示结果
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：后面会指向 Gemini 的 OpenAI 兼容端点
from openai import OpenAI


In [9]:
# ========== 环境：加载并校验 Google API Key ==========

# override=True：.env 中的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)

# 从环境读取 GOOGLE_API_KEY（不要把真实密钥写进笔记本）
api_key = os.getenv("GOOGLE_API_KEY")

# 没有密钥就立刻失败，避免后面请求时才报含糊错误
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env")

# 首尾空白通常来自复制粘贴；密钥含空格会导致鉴权失败
if api_key.strip() != api_key:
    raise ValueError("GOOGLE_API_KEY contains spaces")

# 校验通过后的提示文案保持英文原样
print("GOOGLE_API_KEY loaded successfully")


GOOGLE_API_KEY loaded successfully


In [10]:
# ========== 客户端：用 OpenAI SDK 打 Gemini 兼容端点 ==========

# Gemini 的 OpenAI 兼容 Base URL（字符串保持原样，改了就连不上）
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 创建客户端：base_url 指向 Google，api_key 用上面读到的 GOOGLE_API_KEY
client = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)


In [11]:
# ========== system prompt：规定「Guerilla Search」分析风格 ==========

# 发给模型的系统指令必须保持英文：改译会改变回答风格与行为
system_prompt = """
You are Guerilla Search.

Your goal is NOT to summarize websites.

Your goal is to reveal hidden context and societal undercurrents.

For every website:

1. Explain the official message.
2. Infer possible motivations.
3. Identify missing perspectives.
4. Mention who benefits.
5. Mention groups that may disagree.
6. Point out assumptions or marketing language.
7. Add mild humor and irony.

Never invent facts.
Separate:
- Facts
- Inferences
- Speculation

Respond in markdown.
"""


In [12]:
# ========== user prompt 前缀：告诉模型「要分析的是网站内容」 ==========

# 用户消息前缀保持英文；后面会拼接实际抓取到的网页正文
user_prompt_prefix = """
Analyze this website.

Website Content:
"""


In [13]:
# ========== 组装 messages：system 定角色，user 放内容 ==========

def messages_for(context):
    # 返回 Chat Completions 需要的 messages 列表

    return [
        {
            # system：全局规则（上面的 Guerilla Search 指令）
            "role": "system",
            "content": system_prompt
        },
        {
            # user：前缀说明 + 具体网页上下文
            "role": "user",
            "content": user_prompt_prefix + context
        }
    ]


In [14]:
# ========== 核心流程：抓取网页 → 拼上下文 → 调 Gemini ==========

def summarize(url):
    # 先打印正在抓取的 URL，方便在笔记本里跟踪进度
    print(f"Fetching: {url}")

    # 调用 scraper：把网页正文拉成字符串
    website = fetch_website_contents(url)

    # 再包一层英文 Instructions：约束各分析段落怎么写（影响行为，不翻译）
    context = f"""
WEBSITE CONTENT:

{website}

Instructions:

- Official Story → facts only
- Hidden Context → cautious inference
- Missing Voices → only if noticeable
- Public Reaction → hypothetical
- Never invent events
"""

    # Chat Completions：model id 保持原样；messages 由 messages_for 组装
    response = client.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=messages_for(context)
    )

    # 取出第一条 choice 里的助手回复文本
    return response.choices[0].message.content


In [15]:
# ========== 展示：调用 summarize 并用 Markdown 渲染 ==========

def display_summary(url):
    # 用 try/except 包住，抓取或 API 失败时打印错误而不是整格崩掉
    try:
        # 跑完整分析流水线，得到 markdown 字符串
        result = summarize(url)

        # 在 Jupyter 里漂亮显示 Markdown
        display(Markdown(result))

    except Exception as e:
        # 错误提示标签保持英文原样
        print("Error:")
        print(e)


In [17]:
# ========== 试跑：把 Wikipedia 首页丢进 Guerilla Search ==========

# 改这里的 URL 即可分析别的网站
display_summary("https://wikipedia.org")


Fetching: https://wikipedia.org


Here's a Guerilla Search analysis of the provided Wikipedia content:

### Official Message

Wikipedia presents itself as "The Free Encyclopedia." The provided text highlights the vast number of articles available in numerous languages, with a specific emphasis on the top languages like English, Japanese, German, Russian, French, Spanish, Chinese, Italian, Polish, and Portuguese, each boasting over a million articles. It also lists languages with over 100,000 and 10,000 articles, showcasing its global reach and multilingual accessibility. The implicit message is that Wikipedia is a comprehensive, easily accessible, and globally recognized source of information, freely available to anyone.

### Possible Motivations

The primary motivation behind showcasing these numbers is to reinforce Wikipedia's brand as a dominant and authoritative global knowledge repository. It aims to:

*   **Attract and Retain Users:** By demonstrating its sheer volume and breadth, Wikipedia encourages users to rely on it as their first stop for information.
*   **Attract Contributors:** A large, active community is a sign of a healthy project. Showcasing article counts might inspire potential editors to join and contribute, particularly in languages that might be underrepresented.
*   **Seek Donations:** While not explicitly stated, large organizations often highlight their achievements and scale to solicit financial support. A successful, globally recognized platform is a more compelling recipient of donations.
*   **Maintain Relevance:** In an age of rapidly evolving information platforms, showing continuous growth and broad language support helps Wikipedia maintain its position as a key player.

### Missing Perspectives

*   **Quality vs. Quantity:** The article count metric doesn't speak to the accuracy, depth, or neutrality of the content in each language. A million articles in one language might be vastly different in scope and reliability than a million in another.
*   **Editor Demographics:** Who are these editors? Are they diverse in terms of geography, expertise, and demographics, or is there a concentration of editors from specific regions or backgrounds? This impacts the perspectives reflected in the articles.
*   **Content Gaps:** While impressive, the numbers don't reveal what *isn't* covered. Are there entire fields of knowledge or specific cultural perspectives that are significantly underrepresented across all languages?
*   **Commercial Influence:** The text doesn't touch on how commercial interests, advertising (or lack thereof), and platform policies might shape the content.

### Who Benefits?

*   **The General Public:** Access to a vast, free encyclopedia in their native language is a significant benefit.
*   **Students and Researchers:** Wikipedia serves as a starting point and a quick reference, saving time and effort.
*   **The Wikimedia Foundation:** As the non-profit organization behind Wikipedia, they benefit from the platform's perceived authority and reach, which aids in fundraising and advocacy.
*   **Editors and Contributors:** They benefit from the platform and the community, contributing to their own knowledge and skill development.

### Groups That May Disagree

*   **Traditional Academic Institutions/Publishers:** They might argue that Wikipedia, despite its improvements, can never fully replace peer-reviewed journals or scholarly books for in-depth, authoritative research.
*   **Niche Knowledge Keepers:** Individuals or groups whose specific expertise or cultural knowledge is not adequately represented on Wikipedia might feel its "comprehensiveness" is overstated.
*   **Censorship Advocates (Hypothetical):** While Wikipedia aims for neutrality, its content can be a point of contention for groups who believe certain information should be suppressed or controlled.
*   **Those Critical of Online Information Authority:** Skeptics of crowdsourced information might question the inherent trustworthiness of a platform built by volunteers.

### Assumptions or Marketing Language

*   **"The Free Encyclopedia":** This is a powerful marketing slogan. While the content is free to *access*, the *creation* and *maintenance* are not without cost (server infrastructure, staff, etc.), which are funded by donations. The "free" aspect primarily refers to the licensing and accessibility.
*   **Emphasis on Numbers:** The sheer volume of articles is presented as an indicator of success and quality. This is a common marketing tactic to impress and demonstrate scale. It assumes that more articles automatically equate to a better or more valuable resource.
*   **Global Reach as Inherent Goodness:** The multilingual aspect is framed as universally positive, implying that it inherently promotes understanding and knowledge sharing, without acknowledging potential language-specific biases or challenges.

---

So, Wikipedia is basically saying, "Look how big we are! In *so many languages*! Aren't we amazing?" It's a digital Tower of Babel, but instead of confusion, they promise knowledge. One wonders if the editors in Esperanto feel a bit lonely in their corner, despite having 10,000+ articles. It's a marvel of collective effort, but as always, the devil (and the nuance) is in the details not advertised on the homepage.